# Модуль 05 · Урок 4. Категории, copy-on-write и подготовка графиков

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Categorical для повторяющихся значений
Когда в столбце тысячи повторов из нескольких категорий, тип `category` может значительно сократить память и задать порядок категорий. `astype('category')` хранит список категорий и коды значений. Для упорядоченных категорий (`низкий < средний < высокий`) лучше явно задать `pd.CategoricalDtype(categories=[...],ordered=True)` — иначе лексикографическая сортировка слов не соответствует бизнес-порядку.

In [ ]:
import pandas as pd
levels = pd.Series(['средний','высокий','низкий','средний'])
type_ = pd.CategoricalDtype(['низкий','средний','высокий'],ordered=True)
print(levels.astype(type_).sort_values())

## 2. Индексы, nullable dtype и Copy-on-Write в pandas 3.x
В pandas 3.x Copy-on-Write — основной режим: выборка данных логически независима от исходного DataFrame при последующих изменениях. Не используйте цепное присваивание `df['price'][mask]=0`; вместо него `df.loc[mask,'price']=0`. Строковые типы стали более строгими; нельзя предполагать, что любой текстовый столбец будет иметь `dtype=object`. Для nullable Boolean логика может быть трёхзначной: True, False, NA.

In [ ]:
balances = pd.DataFrame({'amount':[4,0,7]})
mask = balances.amount.eq(0)
balances.loc[mask,'amount'] = 1
print(balances)

## 3. Подготовка данных к визуализации
График должен строиться на данных с подходящей детализацией: для динамики по дням сначала выполняйте группировку/ресемплинг, затем рисуйте линии. `df.plot()` использует Matplotlib; для категориальных сравнений подходит `kind='bar'`, для временного ряда — line, для разброса двух числовых признаков — scatter. Подписи осей, единицы и заголовок делают график читаемым. Избыточные графики не компенсируют неправильную агрегацию.

In [ ]:
import matplotlib.pyplot as plt
traffic = pd.Series([120,145,130],index=['Пн','Вт','Ср'],name='visits')
ax = traffic.plot(kind='bar',title='Посещения за три дня',ylabel='посещений')
plt.tight_layout()
plt.show()

## 4. Выбор уровня детализации и подготовка к отчёту
Группируйте там, где данные имеют понятную бизнес-интерпретацию: среднее время отклика по стратегии, сумма выручки по месяцу, p90 нагрузки по устройству. Не стройте среднее по идентификаторам или сумму по процентам без обоснования. Для повторяемости держите построение таблицы показателей отдельно от рисования.

**Проверьте себя:** зачем ordered categorical? Как правильно менять данные в pandas 3.x? Почему на графике важно указывать единицы измерения?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m5_16 · ordered priority

Получите Series с упорядоченной категориальной шкалой `low`, `medium`, `high` (именно такой порядок), остальные значения должны быть NA.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `labels` — `pd.Series` (индекс и значения определены в ячейке примера).


**Выполнение:** используйте готовые переменные `labels`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_16: используются переменные labels

labels = pd.Series(['medium', 'low', 'high', 'unknown'], index=[0, 1, 2, 3])

print("labels — демонстрационные данные:")

display(labels)


# Снимок правильного результата для автопроверки.
_expected_m5_16 = expected_for("m5_16", labels=labels)


In [ ]:
# m5_16: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: labels
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 4 значений; имя серии: `None`.

```text
0    medium
1       low
2      high
3       NaN
Categories (3, object): ['low' < 'medium' < 'high']
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_16", result, _expected_m5_16)

### m5_17 · chart revenue

Подготовьте Series с месячной суммой price*qty: индекс Period[M], имя Series `revenue`. Результат пригоден для линейного графика.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_17: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m5_17 = expected_for("m5_17", df=df)


In [ ]:
# m5_17: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 2 значений; имя серии: `revenue`.

```text
month
2026-01    889.0
2026-02    488.0
Freq: M
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_17", result, _expected_m5_17)

### m5_18 · chart status

Подготовьте Series частот статусов, расположенную в алфавитном порядке названий статусов, с именем `orders`.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_18: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m5_18 = expected_for("m5_18", df=df)


In [ ]:
# m5_18: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 3 значений; имя серии: `orders`.

```text
status
cancelled    1
new          2
paid         3
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_18", result, _expected_m5_18)

### m5_19 · coerce category

Скопируйте таблицу и приведите `category` к типу category; остальные поля не меняйте.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_19: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m5_19 = expected_for("m5_19", df=df)


In [ ]:
# m5_19: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 10 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment
z  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно
d  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок
x  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                 
q  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_19", result, _expected_m5_19)

### m5_20 · fix cancelled

В копии таблицы замените qty на 0 только для статуса cancelled через `.loc`; исходную таблицу не меняйте.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_20: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m5_20 = expected_for("m5_20", df=df)


In [ ]:
# m5_20: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 10 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment
z  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно
d  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок
x  demo_o5          2   Тверь     кофе   57.0    0        20  cancelled  2026-02-02                 
q  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_20", result, _expected_m5_20)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()